In [6]:
import gymnasium as gym
from taxi_simulator_wrapper import CppTaxiEnv
print(gym.__version__)
env = CppTaxiEnv()


1.2.3


In [4]:
# Importing the necessary libraries and loading the required variables
import numpy as np
import gymnasium as gym
import imageio
from IPython.display import Image
from gymnasium.utils import seeding
from taxi_simulator_wrapper import CppTaxiEnv

# Initializing the C++ Taxi-v3 environment
env = CppTaxiEnv(render_mode='rgb_array')

# Environment seed for reproducibility
env.np_random, _ = seeding.np_random(42)
env.action_space.seed(42)
np.random.seed(42)

# Maximum number of actions per training episode
max_actions = 100 

ModuleNotFoundError: No module named 'gymnasium'

In [ ]:

num_states = env.observation_space.n
num_actions = env.action_space.n
q_table = np.zeros((num_states, num_actions))

# Hyperparameters
num_episodes = 2000
alpha = 0.1
gamma = 1.0
epsilon = 1.0
epsilon_decay = 0.99
min_epsilon = 0.01

# List to store episode returns
episode_returns = []

# Helper functions defined outside loop
def epsilon_greedy(state):
    if np.random.uniform(0,1) < epsilon:
        action = env.action_space.sample()
    else:
        action = np.argmax(q_table[state, :])
    return action

def update_q_table(state, action, reward, new_state):
    old_value = q_table[state, action]
    next_max = np.max(q_table[new_state, :])
    new_value = (1 - alpha) * old_value + alpha * (reward + gamma * next_max)
    q_table[state, action] = new_value

# Training loop
for episode in range(num_episodes):
    state, info = env.reset()
    terminated = False
    truncated = False
    episode_reward = 0

    while not terminated and not truncated:
        action = epsilon_greedy(state)
        new_state, reward, terminated, truncated, info = env.step(action)
        update_q_table(state, action, reward, new_state)
        episode_reward += reward
        state = new_state
    episode_returns.append(episode_reward)
    epsilon = max(min_epsilon, epsilon *  epsilon_decay)

# Learned policy
policy = {state: np.argmax(q_table[state]) for state in range(num_states)}

In [ ]:
# Initializing environment for testing
state, info = env.reset(seed=42)

# Initializing variables for the test episode
frames = []
terminated = False
truncated = False
episode_total_reward = 0
test_actions_count = 0

# Capturing the initial frame
frames.append(env.render())

# Test episode loop
while not terminated and not truncated and test_actions_count < 16:
    action = policy[state]
    new_state, reward, terminated, truncated, info = env.step(action)
    episode_total_reward += reward
    state = new_state
    frames.append(env.render())
    test_actions_count += 1

print(f"The total reward for the episode is: {episode_total_reward}")
print(f"Total steps the agent has taken: {test_actions_count}")
print(f"Number of frames collected: {len(frames)}")

In [ ]:
# Visualizing the agent's behavior through the episode
# Frames saved as a GIF
imageio.mimsave('taxi_agent_behavior.gif', frames, fps=5, loop=0)

# Display GIF
gif_path = "taxi_agent_behavior.gif" 
Image(gif_path) 